# DataportalClient

This notebook exercises `DataportalClient` against the live Material Digital Dataportal at `https://dataportal.material-digital.de`. It creates a private disposable dataset, uploads the repository's Turtle fixture, retrieves the dataset's DCAT RDF representation, waits for triplestore indexing, runs SPARQL queries, and cleans up the created resources.

The DCAT RDF representation describes the CKAN dataset. The uploaded Turtle file contains the domain triples that are expected to become queryable through the generated SPARQL endpoint.

## Environment

Start Jupyter from a shell that exports a Dataportal API token:

```bash
export DATAPORTAL_TOKEN="..."
```

The token must be allowed to create private datasets and resources in at least one organization. The notebook uses the first organization for which CKAN reports `create_dataset` permission.

In [1]:
import os
import time
import uuid
from collections.abc import Mapping
from pathlib import Path

import rdflib


def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None or not value.strip():
        raise RuntimeError(f"Required environment variable {name} is not set.")
    return value.strip()


token = require_env("DATAPORTAL_TOKEN")

In [2]:
from praeco import DataportalClient, Person, PublicationMetadata
from praeco.services.dataportal import DataportalMetadata

client = DataportalClient(api_token=token)
client.base_url

'https://dataportal.material-digital.de'

## Resource Layout

The Dataportal client exposes focused resources for dataset operations, asset operations, DCAT RDF retrieval, and SPARQL queries.

In [3]:
client.datasets, client.assets, client.rdf, client.sparql

(DatasetsResource(client=<praeco.services.dataportal.client.DataportalClient object at 0x741aa04fd6d0>),
 AssetsResource(client=<praeco.services.dataportal.client.DataportalClient object at 0x741aa04fd6d0>),
 RdfResource(client=<praeco.services.dataportal.client.DataportalClient object at 0x741aa04fd6d0>),
 SparqlResource(client=<praeco.services.dataportal.client.DataportalClient object at 0x741aa04fd6d0>))

## Select A Writable Organization

Organization discovery currently uses the underlying CKAN action resource because praeco does not yet expose a Dataportal organization resource. If the token can write to several organizations, adjust `owner_org` before creating the dataset.

In [4]:
organizations = client.action.call(
    "organization_list_for_user",
    {"permission": "create_dataset"},
)
if not isinstance(organizations, list):
    raise RuntimeError("CKAN organization discovery did not return a list.")
writable_organizations = [
    organization for organization in organizations if isinstance(organization, Mapping)
]
if not writable_organizations:
    raise RuntimeError("The token has no organization with create_dataset permission.")

[
    {
        "id": organization.get("id"),
        "name": organization.get("name"),
        "title": organization.get("title"),
    }
    for organization in writable_organizations
]

[{'id': 'd9fe78b4-187e-43ae-80c1-5ac7d20557ad',
  'name': 'mpi-susmat',
  'title': 'MPI-SusMat'},
 {'id': '4db1a5d1-c990-406d-9dbf-a0ca384481d0', 'name': 'pmd', 'title': 'PMD'}]

In [5]:
selected_organization = writable_organizations[0]
owner_org = str(
    selected_organization.get("id") or selected_organization.get("name") or ""
).strip()
if not owner_org:
    raise RuntimeError("The selected organization has neither an id nor a name.")
owner_org

'd9fe78b4-187e-43ae-80c1-5ac7d20557ad'

## Inspect The RDF Fixture

The same small Turtle file used by the Ontodocker notebook is uploaded as a Dataportal asset. Parsing it locally first confirms that the fixture is valid and records the expected triples for the later SPARQL checks.

In [6]:
turtle_path = Path("resources/test_dataset.ttl")
assert turtle_path.is_file(), turtle_path

fixture_graph = rdflib.Graph()
fixture_graph.parse(turtle_path, format="turtle")
EX = rdflib.Namespace("http://example.org/")
assert (EX.alice, EX.worksOn, EX.projectX) in fixture_graph
len(fixture_graph)

13

## Build Dataportal Metadata

`PublicationMetadata` owns service-independent publication fields. `DataportalMetadata` adds CKAN and deployment-specific fields immediately before dataset creation. A random suffix prevents collisions between notebook runs.

In [7]:
suffix = uuid.uuid4().hex[:8]
dataset_name = f"praeco-dataportal-demo-{suffix}"

publication = PublicationMetadata(
    title=f"praeco Dataportal live demo {suffix}",
    description=(
        "Disposable private dataset created by praeco's DataportalClient "
        "integration notebook."
    ),
    creators=(Person(name="Praeco Dataportal Demo"),),
    keywords=("praeco", "dataportal", "integration-test"),
)
metadata = DataportalMetadata(
    metadata=publication,
    name=dataset_name,
    owner_org=owner_org,
    private=True,
    dataset_type="dataset",
)
metadata.to_payload()

{'name': 'praeco-dataportal-demo-11688438',
 'title': 'praeco Dataportal live demo 11688438',
 'notes': "Disposable private dataset created by praeco's DataportalClient integration notebook.",
 'tags': [{'name': 'praeco'},
  {'name': 'dataportal'},
  {'name': 'integration-test'}],
 'owner_org': 'd9fe78b4-187e-43ae-80c1-5ac7d20557ad',
 'private': True,
 'type': 'dataset',
 'extras': [{'key': 'creators',
   'value': '[{"name":"Praeco Dataportal Demo"}]'}]}

## Create And Inspect A Disposable Dataset

The created dataset is kept private. The following cells exercise creation, retrieval, search, and partial metadata updates.

In [8]:
dataset = None
asset = None

dataset = client.datasets.create(metadata)
dataset

DataportalDatasetInfo(id='a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d', name='praeco-dataportal-demo-11688438', title='praeco Dataportal live demo 11688438', notes="Disposable private dataset created by praeco's DataportalClient integration notebook.", owner_org='d9fe78b4-187e-43ae-80c1-5ac7d20557ad', private=True, dataset_type='dataset', raw={'author': None, 'author_email': None, 'creator_user_id': '80c66f6a-727c-4c26-acf5-80194e3556ef', 'id': 'a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d', 'isopen': False, 'license_title': None, 'maintainer': None, 'maintainer_email': None, 'metadata_created': '2026-06-12T08:27:20.370414', 'metadata_modified': '2026-06-12T08:27:20.370418', 'name': 'praeco-dataportal-demo-11688438', 'notes': "Disposable private dataset created by praeco's DataportalClient integration notebook.", 'num_resources': 0, 'num_tags': 3, 'organization': {'id': 'd9fe78b4-187e-43ae-80c1-5ac7d20557ad', 'name': 'mpi-susmat', 'title': 'MPI-SusMat', 'type': 'organization', 'description': '', 'ima

In [20]:
retrieved_dataset = client.datasets.show(dataset)
search_result = client.datasets.search(dataset.name)

assert retrieved_dataset.id == dataset.id
# assert any(result.id == dataset.id for result in search_result.results)
retrieved_dataset

DataportalDatasetInfo(id='a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d', name='praeco-dataportal-demo-11688438', title='praeco Dataportal live demo 11688438', notes='Updated by the praeco Dataportal live demo.', owner_org='d9fe78b4-187e-43ae-80c1-5ac7d20557ad', private=True, dataset_type='dataset', raw={'author': None, 'author_email': None, 'creator_user_id': '80c66f6a-727c-4c26-acf5-80194e3556ef', 'id': 'a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d', 'isopen': False, 'license_title': None, 'maintainer': None, 'maintainer_email': None, 'metadata_created': '2026-06-12T08:27:20.370414', 'metadata_modified': '2026-06-12T09:48:49.707406', 'name': 'praeco-dataportal-demo-11688438', 'notes': 'Updated by the praeco Dataportal live demo.', 'num_resources': 1, 'num_tags': 3, 'organization': {'id': 'd9fe78b4-187e-43ae-80c1-5ac7d20557ad', 'name': 'mpi-susmat', 'title': 'MPI-SusMat', 'type': 'organization', 'description': '', 'image_url': '', 'created': '2025-10-10T08:56:10.061848', 'is_organization': True, 'appr

The outcommented line performs a search across indexed datasets on the instance by leveraging CKANS `package_search`. Here, the assertion will fail, bacause on the dataportal only public datasets are indexed to occure in search results and the dataset created in this demo is private.

In [15]:
dataset = client.datasets.patch(
    dataset,
    {"notes": "Updated by the praeco Dataportal live demo."},
)
dataset.notes

'Updated by the praeco Dataportal live demo.'

## Upload The Turtle Fixture

The Fuseki integration recognizes `format="turtle"` for indexing. The returned asset can be retrieved and patched through the Dataportal-facing resource.

In [24]:
asset = client.assets.upload_rdf(
    dataset,
    turtle_path,
    name="praeco test dataset",
    description="Tiny RDF fixture used by praeco integration notebooks.",
    format="turtle",
)
asset

DataportalAssetInfo(id='bf45829d-bbae-4139-877d-c21e3c67013c', dataset_id='a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d', name='praeco test dataset', description='Tiny RDF fixture used by praeco integration notebooks.', url='https://dataportal.material-digital.de/dataset/a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d/resource/bf45829d-bbae-4139-877d-c21e3c67013c/download/test_dataset.ttl', format='turtle', content_type='text/turtle', size=586, raw={'cache_last_updated': None, 'cache_url': None, 'created': '2026-06-12T10:21:36.926915', 'datastore_active': False, 'description': 'Tiny RDF fixture used by praeco integration notebooks.', 'format': 'turtle', 'hash': '', 'id': 'bf45829d-bbae-4139-877d-c21e3c67013c', 'last_modified': '2026-06-12T10:21:36.910012', 'metadata_modified': '2026-06-12T10:21:36.922178', 'mimetype': 'text/turtle', 'mimetype_inner': None, 'name': 'praeco test dataset', 'package_id': 'a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d', 'position': 1, 'resource_type': None, 'size': 586, 'state': 'acti

In [25]:
retrieved_asset = client.assets.show(asset)
assert retrieved_asset.id == asset.id
assert retrieved_asset.dataset_id == dataset.id
assert retrieved_asset.format is not None
assert retrieved_asset.format.lower() == "turtle"
assert retrieved_asset.content_type == "text/turtle"

asset = client.assets.patch(
    asset,
    {"description": "RDF fixture uploaded and verified by praeco."},
)
asset.description

'RDF fixture uploaded and verified by praeco.'

## Retrieve DCAT RDF

This representation describes the CKAN dataset and its metadata. It is separate from the domain triples in `test_dataset.ttl`.

In [26]:
dcat_turtle = client.rdf.dataset(dataset, format="ttl")
dcat_graph = rdflib.Graph()
dcat_graph.parse(data=dcat_turtle, format="turtle")
assert len(dcat_graph) > 0
len(dcat_graph)

32

## Wait For Triplestore Indexing

The Fuseki update must be requested explicitly and runs asynchronously. The notebook polls its status until the job completes and the deployment exposes the generated `SPARQL` resource. The timeout can be adjusted with `DATAPORTAL_INDEX_TIMEOUT`.

In [27]:
from praeco.exceptions import ValidationError

client.action.call(
    "fuseki_update",
    {
        "pkg_id": dataset.id,
        "resource_ids": [asset.id],
        "persistent": False,
        "reasoning": False,
        "reasoner": "",
    },
)

index_timeout = float(os.getenv("DATAPORTAL_INDEX_TIMEOUT", "180"))
poll_interval = 5.0
deadline = time.monotonic() + index_timeout
sparql_endpoint = None
last_endpoint_error = None
index_status = {}

while time.monotonic() < deadline:
    index_status = client.action.call("fuseki_update_status", {"pkg_id": dataset.id})
    if index_status.get("status") in {"complete", "running_but_viewable"}:
        dataset = client.datasets.show(dataset)
        try:
            sparql_endpoint = client.sparql.endpoint(dataset)
        except ValidationError as exc:
            last_endpoint_error = exc
        else:
            break
    time.sleep(poll_interval)

if sparql_endpoint is None:
    last_endpoint_error = last_endpoint_error or index_status.get("error")
    print(f"SPARQL endpoint was not ready: {last_endpoint_error or index_status}")
else:
    print(sparql_endpoint)

https://dataportal.material-digital.de/dataset/a9e39c2f-9bcf-42fb-81d4-8186d4bfee8d/fuseki/$/sparql


## Query The Uploaded RDF

The queries use identifiers known to be present in `resources/test_dataset.ttl`. Query failures are recorded so cleanup still runs before the notebook reports a validation failure.

In [28]:
validation_errors = []
ask_result = None
frame = None

if sparql_endpoint is None:
    validation_errors.append(
        f"SPARQL endpoint was not available after {index_timeout:g} seconds: "
        f"{last_endpoint_error}"
    )
else:
    try:
        ask_result = client.sparql.query_json(
            sparql_endpoint,
            """
            PREFIX ex: <http://example.org/>
            ASK WHERE { ex:alice ex:worksOn ex:projectX . }
            """,
        )
        if ask_result.get("boolean") is not True:
            raise AssertionError("The expected Alice/projectX triple was not found.")

        frame = client.sparql.query_df(
            sparql_endpoint,
            """
            PREFIX ex: <http://example.org/>
            SELECT ?person ?project
            WHERE { ?person ex:worksOn ?project . }
            ORDER BY ?person ?project
            """,
            columns=["person", "project"],
        )
        expected_row = {
            "person": "http://example.org/alice",
            "project": "http://example.org/projectX",
        }
        if expected_row not in frame.to_dict(orient="records"):
            raise AssertionError("The expected RDF fixture row was not returned.")
    except Exception as exc:
        validation_errors.append(f"SPARQL validation failed: {exc}")

ask_result, frame

({'head': {}, 'boolean': True},
                      person                      project
 0  http://example.org/alice  http://example.org/projectX)

## Cleanup

Cleanup is enabled by default. If execution was interrupted earlier, rerun this cell after restoring `dataset` and `asset` from their displayed identifiers.

In [29]:
cleanup_errors = []

if asset is not None:
    try:
        client.assets.delete(asset)
    except Exception as exc:
        cleanup_errors.append(f"Asset cleanup failed: {exc}")
    else:
        asset = None

if dataset is not None:
    try:
        client.datasets.delete(dataset)
    except Exception as exc:
        cleanup_errors.append(f"Dataset cleanup failed: {exc}")
    else:
        dataset = None

validation_errors.extend(cleanup_errors)
cleanup_errors

[]

## Validation Summary

The final cell fails only after cleanup has been attempted.

In [30]:
if validation_errors:
    raise AssertionError("\n".join(validation_errors))

print("Dataportal live workflow completed successfully.")

Dataportal live workflow completed successfully.
